# Instacart Data Pipeline
## Stage 2: Silver (Cleaning + Validation)
**Source schema:** `workspace.instacart_bronze` | **Target schema:** `workspace.instacart_silver`


### What this notebook does

Builds all 5 Silver tables (`Silver_Aisles`, `Silver_Department`,
`Silver_Products`, `Silver_Orders`, `Silver_Order_Products`), then
runs one consolidated validation pass across all of them before
handing off to Gold. 

### Build Order

| # | Task | Depends on | Why |
|---|---|---|---|
| 1 | `silver_aisles` | `bronze_aisles` | independent |
| 2 | `silver_department` | `bronze_department` | independent |
| 3 | `silver_products` | `silver_aisles`, `silver_department` | checks `EXISTS` against their cleaned output, not Bronze |
| 4 | `silver_orders` | `bronze_order` | independent |
| 5 | `silver_order_products` | `silver_orders` | checks `EXISTS` against `orders_clean` |
| 6 | `silver_validation` | all 5 above | consolidated PASS/REVIEW gate |

🔗 **Job DAG:** rows 3 and 5 above read from another Silver table, not
just Bronze, so in the Job, `silver_products` also depends on
`silver_aisles` + `silver_department`, and `silver_order_products`
also depends on `silver_orders`, on top of the shared
`bronze_validation` gate. Running top-to-bottom in this notebook
reflects that same order directly.

### Design for scale

Every table's build follows the same shape: header, `CREATE OR
REPLACE TABLE ... AS SELECT`, `DESCRIBE TABLE`. Every validation
block follows the same shape too, one `UNION ALL` per table,
producing the same 7 columns. Adding a 6th Silver table later means
copying one cell in each pattern, nothing else needs to change.

### Why `row_difference` doesn't fail validation

This layer intentionally drops rows, orphan FKs, out-of-range
values. A non-zero `row_difference` vs. Bronze is expected, not a
failure. `status` only checks whether problems remain in the Clean
output after those documented drops. See `decisions.md`.

## Part 1: Build

All 5 Silver tables, in dependency order. Each one reads from its
Bronze source, applies trimming/casing/type fixes, and drops rows
that fail a data-quality check: nulls, out-of-range values, or a
foreign key that doesn't resolve.

### silver_aisles

Cleans the aisle lookup table - trims text and standardizes casing.
Smallest and simplest table in this layer.

**Depends on:** nothing else in Silver - safe to build first, in
parallel with every other independent task.

In [0]:
%sql
-- Owner: Ina
-- Name: 09 - Silver Aisles
-- Purpose: Clean the Bronze aisles table - trim whitespace, standardize casing, drop null keys.
-- Grain: One row per aisle, uniquely identified by aisle_id.

CREATE OR REPLACE TABLE aisles_clean AS
SELECT
    aisle_id,
    INITCAP(TRIM(aisle)) AS aisle
FROM instacart_bronze.aisles
WHERE aisle_id IS NOT NULL;

DESCRIBE TABLE aisles_clean;

### silver_department

Cleans the department lookup table - same treatment as aisles, trim
and standardize casing.

**Depends on:** nothing else in Silver.

In [0]:
%sql
-- Owner: Ina
-- Name: 10 - Silver Department
-- Purpose: Clean the Bronze departments table - trim whitespace, standardize casing, drop null keys.
-- Grain: One row per department, uniquely identified by department_id.

CREATE OR REPLACE TABLE departments_clean AS
SELECT
    department_id,
    INITCAP(TRIM(department)) AS department
FROM instacart_bronze.departments
WHERE department_id IS NOT NULL;

DESCRIBE TABLE departments_clean;

### silver_products

Cleans the product catalog - trims names, strips a known backslash
artifact (120 rows carry a stray `\` from a non-standard escaping
pattern where an embedded inch-mark survived ingestion as `\""`
instead of a clean `"`), and drops any product whose aisle or
department doesn't exist in the cleaned lookup tables.

**Depends on:** `silver_aisles`, `silver_department` (already built
above in this notebook) - this query checks `EXISTS` against their
cleaned output, not the raw Bronze tables.

In [0]:
%sql
-- Owner: Ina
-- Name: 11 - Silver Products
-- Purpose: Clean the Bronze products table - trim names, strip stray backslash artifacts,
--          drop rows with an aisle_id or department_id that doesn't exist in Silver.
-- Grain: One row per product, uniquely identified by product_id.
-- CHR(92) represents the backslash character: \

CREATE OR REPLACE TABLE products_clean AS
SELECT
    p.product_id,
    TRIM(REPLACE(p.product_name, CHR(92), '')) AS product_name, 
    p.aisle_id,
    p.department_id
FROM instacart_bronze.products p
WHERE p.product_id IS NOT NULL
  AND p.product_name IS NOT NULL
  AND EXISTS (SELECT 1 FROM aisles_clean a WHERE a.aisle_id = p.aisle_id)
  AND EXISTS (SELECT 1 FROM departments_clean d WHERE d.department_id = p.department_id);

DESCRIBE TABLE products_clean;

### silver_orders

Cleans the orders table - drops null keys and validates that
`order_dow` and `order_hour_of_day` fall in a sane range.
`days_since_prior_order` NULLs are kept on purpose, since NULL means
a user's first order, not missing data.

**Depends on:** nothing else in Silver - but `silver_order_products`
below depends on this one, so it needs to finish first.

In [0]:
%sql
-- Owner: Ina
-- Name: 12 - Silver Orders
-- Purpose: Clean the Bronze orders table - drop null keys, validate day-of-week and hour ranges.
-- Grain: One row per order, uniquely identified by order_id.

CREATE OR REPLACE TABLE orders_clean AS
SELECT
    order_id,
    user_id,
    order_number,
    order_dow,
    order_hour_of_day,
    days_since_prior_order          -- NULL preserved: meaningful for a user's first order
FROM instacart_bronze.orders
WHERE order_id IS NOT NULL
  AND user_id IS NOT NULL
  AND order_dow BETWEEN 0 AND 6
  AND order_hour_of_day BETWEEN 0 AND 23;

DESCRIBE TABLE orders_clean;

### silver_order_products

Unions the two Bronze source tables (`order_products_prior`,
`order_products_train`) into one table, drops rows whose order
doesn't exist in Silver, and casts `reordered` to boolean. The
prior/train split is a Kaggle ML-competition artifact, not a real
business distinction - see `decisions.md`.

**Depends on:** `silver_orders` (already built above) - this query
checks `EXISTS` against `orders_clean`, not Bronze `orders`.

In [0]:
%sql
-- Owner: Ina
-- Name: 13 - Silver Order Products
-- Purpose: Union the Bronze order_products_prior and order_products_train tables into
--          one, drop rows whose order_id doesn't exist in Silver, cast reordered to boolean.
-- Grain: One row per product line in one order, uniquely identified by (order_id, product_id).

CREATE OR REPLACE TABLE order_products_clean AS
WITH order_products_combined AS (
    SELECT *, 'prior' AS source_file
    FROM instacart_bronze.order_products_prior
    UNION ALL
    SELECT *, 'train' AS source_file
    FROM instacart_bronze.order_products_train
)
SELECT
    op.order_id,
    op.product_id,
    op.add_to_cart_order,
    CAST(op.reordered AS BOOLEAN) AS reordered,
    op.source_file
FROM order_products_combined op
WHERE op.order_id IS NOT NULL
  AND op.product_id IS NOT NULL
  AND EXISTS (SELECT 1 FROM products_clean p WHERE p.product_id = op.product_id);

DESCRIBE TABLE order_products_clean;

-- source_file breakdown (informational) - confirms the union looks right
SELECT source_file, COUNT(*) AS row_count
FROM order_products_clean
GROUP BY source_file;

## Part 2: Validate

One consolidated check across all 5 tables built above.
`status = 'PASS'` only when every check for that table is clean.

In [0]:
%sql
-- Owner: Ina
-- Name: 14 - Silver Validation
-- Purpose: Validate Silver row counts, required identifiers, candidate keys, required fields,
--          referential integrity, and text-quality artifacts with table-level pass or fail results.
-- Grain: One validation summary row per Silver table.

WITH validation AS (

    SELECT
        'aisles' AS table_name,
        (SELECT COUNT(*) FROM instacart_bronze.aisles) AS raw_rows,
        COUNT(*) AS clean_rows,
        COUNT(*) - (SELECT COUNT(*) FROM instacart_bronze.aisles) AS row_difference,
        SUM(CASE WHEN aisle_id IS NULL THEN 1 ELSE 0 END) AS null_key_rows,
        (SELECT COUNT(*) FROM (
            SELECT aisle_id FROM aisles_clean
            WHERE aisle_id IS NOT NULL GROUP BY aisle_id HAVING COUNT(*) > 1
        )) AS duplicate_keys,
        SUM(CASE WHEN aisle IS NULL OR TRIM(aisle) = '' THEN 1 ELSE 0 END) AS required_field_issues,
        0 AS unmatched_fk_rows
    FROM aisles_clean

    UNION ALL

    SELECT
        'departments',
        (SELECT COUNT(*) FROM instacart_bronze.departments),
        COUNT(*),
        COUNT(*) - (SELECT COUNT(*) FROM instacart_bronze.departments),
        SUM(CASE WHEN department_id IS NULL THEN 1 ELSE 0 END),
        (SELECT COUNT(*) FROM (
            SELECT department_id FROM departments_clean
            WHERE department_id IS NOT NULL GROUP BY department_id HAVING COUNT(*) > 1
        )),
        SUM(CASE WHEN department IS NULL OR TRIM(department) = '' THEN 1 ELSE 0 END),
        0
    FROM departments_clean

    UNION ALL

    SELECT
        'products',
        (SELECT COUNT(*) FROM instacart_bronze.products),
        COUNT(*),
        COUNT(*) - (SELECT COUNT(*) FROM instacart_bronze.products),
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END),
        (SELECT COUNT(*) FROM (
            SELECT product_id FROM products_clean
            WHERE product_id IS NOT NULL GROUP BY product_id HAVING COUNT(*) > 1
        )),
        SUM(CASE WHEN product_name IS NULL OR TRIM(product_name) = '' THEN 1 ELSE 0 END)
         + SUM(CASE WHEN INSTR(product_name, CHR(92)) > 0 THEN 1 ELSE 0 END),
        (SELECT COUNT(*) FROM products_clean p
            LEFT JOIN aisles_clean a ON p.aisle_id = a.aisle_id
            WHERE p.aisle_id IS NOT NULL AND a.aisle_id IS NULL)
         + (SELECT COUNT(*) FROM products_clean p
            LEFT JOIN departments_clean d ON p.department_id = d.department_id
            WHERE p.department_id IS NOT NULL AND d.department_id IS NULL)
    FROM products_clean

    UNION ALL

    SELECT
        'orders',
        (SELECT COUNT(*) FROM instacart_bronze.orders),
        COUNT(*),
        COUNT(*) - (SELECT COUNT(*) FROM instacart_bronze.orders),
        SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END),
        (SELECT COUNT(*) FROM (
            SELECT order_id FROM orders_clean
            WHERE order_id IS NOT NULL GROUP BY order_id HAVING COUNT(*) > 1
        )),
        SUM(CASE
            WHEN user_id IS NULL
              OR order_number IS NULL
              OR order_dow NOT BETWEEN 0 AND 6
              OR order_hour_of_day NOT BETWEEN 0 AND 23
            
            -- First order should have no prior order interval
              OR (
                  order_number = 1
                  AND days_since_prior_order IS NOT NULL
              )

              -- Orders after the first should have a prior order interval
              OR (
                  order_number > 1
                  AND days_since_prior_order IS NULL
              )
            THEN 1
            ELSE 0
        END
    ),
    0
    FROM orders_clean

    UNION ALL

    SELECT
        'order_products',
        (SELECT COUNT(*) FROM instacart_bronze.order_products_prior)
          + (SELECT COUNT(*) FROM instacart_bronze.order_products_train),
        COUNT(*),
        COUNT(*) - (
            (SELECT COUNT(*) FROM instacart_bronze.order_products_prior)
          + (SELECT COUNT(*) FROM instacart_bronze.order_products_train)
        ),
        SUM(CASE WHEN order_id IS NULL OR product_id IS NULL THEN 1 ELSE 0 END),
        (SELECT COUNT(*) FROM (
            SELECT order_id, product_id FROM order_products_clean
            GROUP BY order_id, product_id HAVING COUNT(*) > 1
        )),
        SUM(CASE
            WHEN add_to_cart_order IS NULL OR add_to_cart_order <= 0
            THEN 1 ELSE 0
        END),
        (SELECT COUNT(*) FROM order_products_clean op
            LEFT JOIN orders_clean o ON op.order_id = o.order_id
            WHERE o.order_id IS NULL)
         + (SELECT COUNT(*) FROM order_products_clean op
            LEFT JOIN products_clean p ON op.product_id = p.product_id
            WHERE p.product_id IS NULL)
    FROM order_products_clean

)

SELECT
    *,
    CASE
        WHEN null_key_rows = 0
             AND duplicate_keys = 0
             AND required_field_issues = 0
             AND unmatched_fk_rows = 0
        THEN 'PASS'
        ELSE 'REVIEW'
    END AS status
FROM validation
ORDER BY table_name;

## Silver Layer: Summary

| Table | Status | Notes |
|---|---|---|
| `aisles_clean` | Built + validated | Simple trim/casing, no dependencies |
| `departments_clean` | Built + validated | Simple trim/casing, no dependencies |
| `products_clean` | Built + validated | Depends on aisles + department, backslash fix applied |
| `orders_clean` | Built + validated | NULL `days_since_prior_order` preserved intentionally |
| `order_products_clean` | Built + validated | Depends on orders, unions prior + train |

- `row_difference` is shown per table but does not affect `status` -
  see the design note at the top of this notebook.
- `products`' `required_field_issues` also counts any remaining
  backslash matches - a regression in the fix surfaces as `REVIEW`
  automatically, not as a separate manual check.

**Expected result:** `status = 'PASS'` on all 5 rows.

**Next stage:** Gold layer - build `dim_order`, `dim_product`, and
`fact_order_products` from `workspace.instacart_silver`.